# Vibration Prediction System - Example Usage

This notebook demonstrates how to use the vibration prediction system for training and evaluating LSTM models on vibration data.

## 1. Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from pathlib import Path

# Add project to path
project_root = Path.cwd()
sys.path.append(str(project_root))

# Import project modules
from data.dataset import VibrationDataset, create_dataloaders
from models.lstm_predictor import LSTMPredictor
from training.trainer import Trainer
from utils.visualization import VibrationVisualizer
from utils.signal_processing import compute_spectral_features, detect_bifurcation_points

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete!")

## 2. Load Configuration

In [ ]:
# Load configuration files
with open('config/base_config.yaml', 'r') as f:
    base_config = yaml.safe_load(f)

with open('config/lstm_config.yaml', 'r') as f:
    lstm_config = yaml.safe_load(f)

# Merge configurations
config = base_config.copy()
for key, value in lstm_config.items():
    if key in config and isinstance(config[key], dict) and isinstance(value, dict):
        config[key].update(value)
    else:
        config[key] = value

print("Configuration loaded:")
print(f"- Window size: {config['data']['window_size']}")
print(f"- Prediction horizon: {config['data']['prediction_horizon']}")
print(f"- Model type: {config['model']['type']}")
print(f"- Hidden size: {config['model']['hidden_size']}")

## 3. Load and Explore Dataset

In [ ]:
# Create dataset
dataset_config = {
    'data_path': 'create_datasets/datasets/mathieu_delayed_dataset.pkl',
    'window_size': config['data']['window_size'],
    'prediction_horizon': config['data']['prediction_horizon'],
    'stride': config['data']['stride'],
    'train_split': config['data']['train_split'],
    'val_split': config['data']['val_split'],
    'test_split': config['data']['test_split'],
    'sampling_rate': config['data']['sampling_rate'],
    'features': config['features']['use_features'],
    'random_seed': config['random_seed']
}

# Create train dataset to explore
train_dataset = VibrationDataset(split='train', **dataset_config)

print(f"Dataset loaded:")
print(f"- Total sequences: {len(train_dataset)}")
print(f"- Feature names: {train_dataset.features}")

# Get a sample
sample = train_dataset[0]
print(f"\nSample shapes:")
print(f"- Features: {sample['features'].shape}")
print(f"- Parameters: {sample['parameters'].shape}")
print(f"- Targets: {sample['targets'].shape}")
print(f"- Max amplitude: {sample['max_amplitude'].item():.4f}")

## 4. Visualize Sample Data

In [ ]:
# Create visualizer
visualizer = VibrationVisualizer()

# Get sample data
sample = train_dataset[0]
features = sample['features'].numpy()
targets = sample['targets'].numpy()

# Plot trajectory comparison (using features as input, targets as ground truth)
fig = visualizer.plot_trajectory_comparison(
    input_data=features,
    target_data=targets,
    predicted_data=targets,  # Using targets as "prediction" for visualization
    feature_names=['Position', 'Velocity'],
    title='Sample Vibration Trajectory'
)
plt.show()

# Plot phase space
fig = visualizer.plot_phase_space(
    trajectories=[features, targets],
    labels=['Input', 'Target'],
    title='Phase Space Plot'
)
plt.show()

## 5. Analyze Signal Properties

In [ ]:
# Analyze spectral properties
position_signal = features[:, 0]
spectral_features = compute_spectral_features(position_signal, fs=config['data']['sampling_rate'])

print("Spectral Features:")
print(f"- Dominant frequency: {spectral_features['dominant_frequency']:.2f} Hz")
print(f"- Spectral centroid: {spectral_features['spectral_centroid']:.2f} Hz")
print(f"- Spectral spread: {spectral_features['spectral_spread']:.2f} Hz")
print(f"- Total power: {spectral_features['total_power']:.4f}")

# Detect bifurcation points
bifurcation_points, bifurcation_amplitudes = detect_bifurcation_points(position_signal)
print(f"\nBifurcation Analysis:")
print(f"- Number of bifurcation points: {len(bifurcation_points)}")
if len(bifurcation_points) > 0:
    print(f"- Max bifurcation amplitude: {np.max(bifurcation_amplitudes):.4f}")

# Plot frequency analysis
fig = visualizer.plot_frequency_analysis(
    signals=[position_signal],
    labels=['Position'],
    sampling_rate=config['data']['sampling_rate'],
    title='Frequency Domain Analysis'
)
plt.show()

## 6. Create Data Loaders

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader = create_dataloaders(
    dataset_config,
    batch_size=config['training']['batch_size'],
    num_workers=2  # Reduce for notebook
)

print(f"Data loaders created:")
print(f"- Train batches: {len(train_loader)}")
print(f"- Validation batches: {len(val_loader)}")
print(f"- Test batches: {len(test_loader)}")

# Check a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"- {key}: {value.shape}")
    else:
        print(f"- {key}: {type(value)}")

## 7. Create and Initialize Model

In [ ]:
# Setup model configuration
model_config = config['model'].copy()
model_config['input_size'] = 2  # x, x_dot
model_config['output_size'] = 2  # x, x_dot
model_config['n_params'] = 15  # Number of system parameters

# Create model
model = LSTMPredictor(model_config)

# Print model info
model_info = model.get_model_info()
print(f"Model Information:")
print(f"- Type: {model_info['model_type']}")
print(f"- Total parameters: {model_info['total_parameters']:,}")
print(f"- Trainable parameters: {model_info['trainable_parameters']:,}")

# Test forward pass
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

# Move batch to device
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

# Forward pass
with torch.no_grad():
    outputs = model(batch['features'], batch['parameters'])

print(f"\nModel outputs:")
for key, value in outputs.items():
    if isinstance(value, torch.Tensor):
        print(f"- {key}: {value.shape}")

print(f"\nUsing device: {device}")

## 8. Train Model (Short Demo)

In [ ]:
# Create a short training configuration for demo
demo_config = config['training'].copy()
demo_config['epochs'] = 5  # Short demo
demo_config['log_interval'] = 1
demo_config['early_stopping_patience'] = 10
demo_config['output_dir'] = 'demo_outputs'

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=demo_config,
    device=device
)

print("Starting demo training (5 epochs)...")

# Train for a few epochs
results = trainer.train()

print(f"\nDemo training completed!")
print(f"- Best validation loss: {results['best_val_loss']:.6f}")
print(f"- Training time: {results['training_time']:.2f} seconds")
print(f"- Total epochs: {results['total_epochs']}")

## 9. Evaluate Model Performance

In [ ]:
# Evaluate on test set
from training.metrics import VibrationMetrics

model.eval()
metrics = VibrationMetrics()

all_predictions = []
all_targets = []
all_amplitudes_pred = []
all_amplitudes_true = []

print("Evaluating model on test set...")

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= 10:  # Limit for demo
            break
            
        # Move to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                for k, v in batch.items()}
        
        # Forward pass
        outputs = model(batch['features'], batch['parameters'])
        
        # Collect results
        all_predictions.append(outputs['trajectory'].cpu().numpy())
        all_targets.append(batch['targets'].cpu().numpy())
        
        if 'amplitude' in outputs:
            all_amplitudes_pred.append(outputs['amplitude'].cpu().numpy())
        if 'max_amplitude' in batch:
            all_amplitudes_true.append(batch['max_amplitude'].cpu().numpy())
        
        # Update metrics
        metrics.update(
            predictions=outputs['trajectory'],
            targets=batch['targets'],
            amplitudes_pred=outputs.get('amplitude'),
            amplitudes_true=batch.get('max_amplitude')
        )

# Compute metrics
evaluation_results = metrics.compute_all_metrics()

# Print summary
print("\nEvaluation Results:")
summary = metrics.get_summary_metrics()
for metric_name, value in summary.items():
    print(f"- {metric_name}: {value:.6f}")

## 10. Visualize Predictions

In [ ]:
# Concatenate results for visualization
predictions = np.concatenate(all_predictions, axis=0)
targets = np.concatenate(all_targets, axis=0)

# Plot sample predictions
n_samples = min(3, len(predictions))

for i in range(n_samples):
    # Create dummy input (use first part of target)
    input_data = targets[i, :config['data']['window_size']//2, :]
    target_data = targets[i, config['data']['window_size']//2:, :]
    predicted_data = predictions[i]
    
    fig = visualizer.plot_trajectory_comparison(
        input_data,
        target_data,
        predicted_data,
        title=f'Sample {i+1} - Prediction vs Ground Truth'
    )
    plt.show()

# Plot amplitude analysis if available
if all_amplitudes_pred and all_amplitudes_true:
    amplitudes_pred = np.concatenate(all_amplitudes_pred, axis=0)
    amplitudes_true = np.concatenate(all_amplitudes_true, axis=0)
    
    fig = visualizer.plot_amplitude_analysis(
        amplitudes_pred.flatten(),
        amplitudes_true.flatten(),
        title='Amplitude Prediction Analysis'
    )
    plt.show()

## 11. Attention Analysis (if available)

In [ ]:
# Get attention weights for a sample
sample_batch = next(iter(test_loader))
sample_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
               for k, v in sample_batch.items()}

with torch.no_grad():
    # Get attention weights
    attention_weights = model.get_attention_weights(
        sample_batch['features'][:1],  # First sample only
        sample_batch['parameters'][:1]
    )

if attention_weights.numel() > 0:
    # Plot attention weights
    fig = visualizer.plot_attention_weights(
        attention_weights.cpu().numpy(),
        sample_batch['features'][0].cpu().numpy(),
        title='Attention Weights Visualization'
    )
    plt.show()
else:
    print("No attention weights available for this model.")

## 12. Model Performance Summary

In [ ]:
# Create performance summary visualization
fig = visualizer.plot_model_performance_summary(
    evaluation_results,
    title='Model Performance Summary'
)
plt.show()

# Print detailed results
print("\nDetailed Evaluation Results:")
print("=" * 50)

# Group metrics by category
trajectory_metrics = {k: v for k, v in evaluation_results.items() 
                     if any(x in k.lower() for x in ['rmse', 'mae', 'r2'])}
amplitude_metrics = {k: v for k, v in evaluation_results.items() 
                    if 'amplitude' in k.lower()}
bifurcation_metrics = {k: v for k, v in evaluation_results.items() 
                      if 'bifurcation' in k.lower()}

if trajectory_metrics:
    print("\nTrajectory Metrics:")
    for key, value in trajectory_metrics.items():
        print(f"  {key}: {value:.6f}")

if amplitude_metrics:
    print("\nAmplitude Metrics:")
    for key, value in amplitude_metrics.items():
        print(f"  {key}: {value:.6f}")

if bifurcation_metrics:
    print("\nBifurcation Detection Metrics:")
    for key, value in bifurcation_metrics.items():
        print(f"  {key}: {value:.6f}")

print("\n" + "=" * 50)
print("Demo completed successfully!")
print("\nFor full training, use the command line interface:")
print("python main.py --mode train --config config/base_config.yaml --model-config config/lstm_config.yaml")